<a href="https://colab.research.google.com/github/Zeldano118/QPon_NLP_PBA/blob/main/notebooks/01_scraping_tokenization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# QPon Review Scraping + NLP Tokenization

| | |
|---|---|
| **App** | [QPon](https://play.google.com/store/apps/details?id=com.qpon.platform) (`com.qpon.platform`) |
| **Author** | Zeldano Shan Oeffie (5026231118) |
| **Course** | Natural Language Processing |

This notebook covers data collection and initial NLP exploration for the QPon sentiment analysis project:
1. Scrape all QPon reviews from Google Play
2. Tokenize English text (Gutenberg) and Indonesian text (IndoNLU) as baselines
3. Tokenize, stem, and remove stopwords from QPon reviews
4. Save raw data for preprocessing in the next notebook

In [14]:
!pip install google-play-scraper PySastrawi nltk datasets -q

In [15]:
import pandas as pd
import numpy as np
from google_play_scraper import reviews_all, Sort
from google_play_scraper import app as app_info
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from datasets import load_dataset
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from collections import Counter
import time, os

nltk.download('gutenberg', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

True

---
## A. Scraping QPon Reviews

In [16]:
info = app_info('com.qpon.platform', lang='id', country='id')

print(f"App        : {info['title']}")
print(f"Developer  : {info['developer']}")
print(f"Rating     : {info['score']:.2f}  ({info['reviews']:,} reviews)")
print(f"Installs   : {info['installs']}")
print(f"Version    : {info['version']}")

App        : Qpon-Selalu Ada Diskon
Developer  : QPON DIGITAL SINGAPORE
Rating     : 3.51  (5,217 reviews)
Installs   : 10.000.000+
Version    : 2.23.0.0


In [17]:
print('Scraping all QPon reviews — this takes a few minutes...\n')
t0 = time.time()

qpon_reviews = reviews_all(
    'com.qpon.platform',
    lang='id',
    country='id',
    sort=Sort.NEWEST,
    sleep_milliseconds=100
)

print(f'Done — {len(qpon_reviews):,} reviews in {time.time()-t0:.0f}s')

Scraping all QPon reviews — this takes a few minutes...

Done — 4,659 reviews in 2s


In [18]:
df = pd.DataFrame(qpon_reviews)
print(f'{len(df):,} reviews, {df.shape[1]} columns')
print(f'Columns: {df.columns.tolist()}\n')
df.dtypes

4,659 reviews, 11 columns
Columns: ['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']



,0
reviewId,object
userName,object
userImage,object
content,object
score,int64
thumbsUpCount,int64
reviewCreatedVersion,object
at,datetime64[ns]
replyContent,object
repliedAt,datetime64[ns]


In [19]:
df.to_csv('qpon_reviews_raw.csv', index=False)
print(f'Saved — {os.path.getsize("qpon_reviews_raw.csv")/1024:.0f} KB')

Saved — 2134 KB


In [20]:
# quick look at the data
for _, r in df.head(5).iterrows():
    stars = '\u2B50' * r['score']
    print(f"{stars}  {r['userName']}")
    print(f"   {str(r['content'])[:150]}")
    print()

⭐⭐⭐⭐⭐  Pengguna Google
   keren

⭐  Pengguna Google
   tukar vocer qpon saldo udah kepotong status belum bayar, jadi 2x bayar... mohon ditindak lanjut biar gak ada yang seperti itu lagi

⭐  Pengguna Google
   promonya kurang masih mahal, murahan jga tiktok

⭐⭐⭐⭐⭐  Pengguna Google
   very good .. all in one..

⭐  Pengguna Google
   UNINSTALL AJA GUYS MAKIN LAMA MAKIN JELEK, MAKIN ANEH, MASA DAPAT VOUCHER 5RB TERUS BESOK CHECK-IN LAGI BERUBAH JADI CUMA DAPAT 3 RB AWALNYA DOANG BAG



In [21]:
print('Rating distribution:\n')
dist = df['score'].value_counts().sort_index()
for score, n in dist.items():
    pct = n / len(df) * 100
    bar = '\u2588' * int(pct / 2)
    print(f'  {score}\u2B50  {n:>5,}  ({pct:4.1f}%)  {bar}')
print(f'\nAverage: {df["score"].mean():.2f}')

Rating distribution:

  1⭐  2,046  (43.9%)  █████████████████████
  2⭐    269  ( 5.8%)  ██
  3⭐    169  ( 3.6%)  █
  4⭐    130  ( 2.8%)  █
  5⭐  2,045  (43.9%)  █████████████████████

Average: 2.97


In [22]:
for score in range(1, 6):
    subset = df[df['score'] == score]['content'].dropna()
    if len(subset) == 0: continue
    print(f"\n{'='*50}")
    print(f"Rating {score}/5 — {len(subset):,} reviews")
    print(f"{'='*50}")
    for txt in subset.head(2):
        print(f"  {str(txt)[:180]}")


Rating 1/5 — 2,046 reviews
  tukar vocer qpon saldo udah kepotong status belum bayar, jadi 2x bayar... mohon ditindak lanjut biar gak ada yang seperti itu lagi
  promonya kurang masih mahal, murahan jga tiktok

Rating 2/5 — 269 reviews
  br mau login, malah gbisa nih.. ket nya akun terdeteksi tdk normal.. nyoba login lg d tgl 8/3/26 tetap gbisa wkwk pas tgl 02 kmrn chat cs nya dsruh cb lg besok udh brpa hri ttp gbi
  gak bagus

Rating 3/5 — 169 reviews
  karena pas flash sale yang di liat hanya brand aja alhasil gak bisa kepake dan jeleknya GAK ADA OPTION REFUND . saya di bandung salah beli voucher malah yang jakarta .. tolong dong
  kenapa sekarang update terbaru malah tenant² nya gak sesuai lokasi kita ya?, biasanya merunut dr yg terdekat, ini langsung merchant ibu kota aja. Dan gak ada lagi sort berdasarkan 

Rating 4/5 — 130 reviews
  saeee
  mantap diskon nya

Rating 5/5 — 2,045 reviews
  keren
  very good .. all in one..


---
## B. Baseline — English Tokenization (Gutenberg)
Quick tokenization on formal English text as a baseline before moving to Indonesian.

In [23]:
text_en = nltk.corpus.gutenberg.raw('austen-emma.txt')[:1000]
tokens_en = word_tokenize(text_en)

print('First 500 chars:')
print(text_en[:500])
print(f'\n→ {len(tokens_en)} tokens, {len(set(tokens_en))} unique')

First 500 chars:
[Emma by Jane Austen 1816]

VOLUME I

CHAPTER I


Emma Woodhouse, handsome, clever, and rich, with a comfortable home
and happy disposition, seemed to unite some of the best blessings
of existence; and had lived nearly twenty-one years in the world
with very little to distress or vex her.

She was the youngest of the two daughters of a most affectionate,
indulgent father; and had, in consequence of her sister's marriage,
been mistress of his house from a very early period.  Her mother
had died t

→ 198 tokens, 114 unique


---
## C. Baseline — Indonesian Tokenization (IndoNLU SMSA)
Tokenizing Indonesian social media text from the [IndoNLU SMSA](https://huggingface.co/datasets/kornwtp/indonlu-smsa) dataset. Closer to QPon reviews in style — short, informal, lots of slang.

In [24]:
dataset = load_dataset('kornwtp/indonlu-smsa', split='train')
df_indo = dataset.to_pandas()
print(f'{len(df_indo)} rows, columns: {list(df_indo.columns)}\n')

text_col = df_indo.columns[0]
label_col = df_indo.columns[1]
samples = df_indo[[text_col, label_col]].head(3)

stemmer = StemmerFactory().create_stemmer()

for i, (_, row) in enumerate(samples.iterrows()):
    text = str(row[text_col])
    tokens = word_tokenize(text)
    stemmed = word_tokenize(stemmer.stem(text))

    print(f'Sample {i+1} (label={row[label_col]}):')
    print(f'  Original : {text}')
    print(f'  Tokens   : {tokens}')
    print(f'  Stemmed  : {stemmed}')
    print(f'  {len(tokens)} tokens → {len(stemmed)} after stemming\n')

    # save token files
    with open(f'tokens_sample_{i+1}.txt', 'w') as f:
        f.write('\n'.join(tokens))

# top words across samples
all_tok = []
for t in samples[text_col]:
    all_tok.extend(word_tokenize(str(t).lower()))
print('Top 10:', Counter(all_tok).most_common(10))

11000 rows, columns: ['texts', 'labels']

Sample 1 (label=0):
  Original : warung ini dimiliki oleh pengusaha pabrik tahu yang sudah puluhan tahun terkenal membuat tahu putih di bandung . tahu berkualitas , dipadu keahlian memasak , dipadu kretivitas , jadilah warung yang menyajikan menu utama berbahan tahu , ditambah menu umum lain seperti ayam . semuanya selera indonesia . harga cukup terjangkau . jangan lewatkan tahu bletoka nya , tidak kalah dengan yang asli dari tegal !
  Tokens   : ['warung', 'ini', 'dimiliki', 'oleh', 'pengusaha', 'pabrik', 'tahu', 'yang', 'sudah', 'puluhan', 'tahun', 'terkenal', 'membuat', 'tahu', 'putih', 'di', 'bandung', '.', 'tahu', 'berkualitas', ',', 'dipadu', 'keahlian', 'memasak', ',', 'dipadu', 'kretivitas', ',', 'jadilah', 'warung', 'yang', 'menyajikan', 'menu', 'utama', 'berbahan', 'tahu', ',', 'ditambah', 'menu', 'umum', 'lain', 'seperti', 'ayam', '.', 'semuanya', 'selera', 'indonesia', '.', 'harga', 'cukup', 'terjangkau', '.', 'jangan', 'lewatkan', 

---
## D. Tokenization on QPon Reviews
Applying the same pipeline to the actual QPon data.

In [25]:
samples_qpon = df['content'].dropna().head(5).tolist()

for i, rev in enumerate(samples_qpon):
    toks = word_tokenize(str(rev))
    print(f'Review {i+1}:')
    print(f'  Text   : {rev}')
    print(f'  Tokens : {toks}')
    print(f'  Count  : {len(toks)}\n')

# frequency across entire dataset
all_qpon = []
for rev in df['content'].dropna():
    all_qpon.extend(word_tokenize(str(rev).lower()))

freq = Counter(all_qpon)
print(f'Total: {len(all_qpon):,} tokens, {len(set(all_qpon)):,} unique\n')
print('Most frequent:')
for w, c in freq.most_common(15):
    print(f'  {w:20s} {c:>6,}')

Review 1:
  Text   : keren
  Tokens : ['keren']
  Count  : 1

Review 2:
  Text   : tukar vocer qpon saldo udah kepotong status belum bayar, jadi 2x bayar... mohon ditindak lanjut biar gak ada yang seperti itu lagi
  Tokens : ['tukar', 'vocer', 'qpon', 'saldo', 'udah', 'kepotong', 'status', 'belum', 'bayar', ',', 'jadi', '2x', 'bayar', '...', 'mohon', 'ditindak', 'lanjut', 'biar', 'gak', 'ada', 'yang', 'seperti', 'itu', 'lagi']
  Count  : 24

Review 3:
  Text   : promonya kurang masih mahal, murahan jga tiktok
  Tokens : ['promonya', 'kurang', 'masih', 'mahal', ',', 'murahan', 'jga', 'tiktok']
  Count  : 8

Review 4:
  Text   : very good .. all in one..
  Tokens : ['very', 'good', '..', 'all', 'in', 'one', '..']
  Count  : 7

Review 5:
  Text   : UNINSTALL AJA GUYS MAKIN LAMA MAKIN JELEK, MAKIN ANEH, MASA DAPAT VOUCHER 5RB TERUS BESOK CHECK-IN LAGI BERUBAH JADI CUMA DAPAT 3 RB AWALNYA DOANG BAGUS TERUS HARGA ASLI SEBELUM DISKON MALAH LEBIH MAHAL DARI OFFSTORE, DITANYA KE CS GA TAU JAWAB

In [26]:
# stemming on frequent long words from QPon
long_w = [w for w in set(all_qpon) if len(w) > 6 and w.isalpha()]
long_w = sorted(long_w, key=lambda w: freq[w], reverse=True)[:12]

print('Stemming demo:\n')
for w in long_w:
    s = stemmer.stem(w)
    mark = ' ←' if s != w else ''
    print(f'  {w:22s} ({freq[w]:,}x)  →  {s}{mark}')

# stopword removal
stop = set(stopwords.words('indonesian'))
print(f'\nStopword removal ({len(stop)} Indonesian stopwords):\n')

for i, rev in enumerate(samples_qpon[:3]):
    toks = word_tokenize(str(rev).lower())
    clean = [w for w in toks if w not in stop and w.isalpha()]
    print(f'Review {i+1}:')
    print(f'  Before : {toks}')
    print(f'  After  : {clean}')
    print(f'  Removed {len(toks)-len(clean)} tokens\n')

Stemming demo:

  aplikasi               (1,417x)  →  aplikasi
  voucher                (797x)  →  voucher
  makanan                (320x)  →  makan ←
  padahal                (293x)  →  padahal
  membantu               (219x)  →  bantu ←
  gampang                (216x)  →  gampang
  download               (185x)  →  download
  dipakai                (183x)  →  pakai ←
  aplikasinya            (166x)  →  aplikasi ←
  restoran               (143x)  →  restoran
  langsung               (125x)  →  langsung
  terlalu                (118x)  →  terlalu

Stopword removal (757 Indonesian stopwords):

Review 1:
  Before : ['keren']
  After  : ['keren']
  Removed 0 tokens

Review 2:
  Before : ['tukar', 'vocer', 'qpon', 'saldo', 'udah', 'kepotong', 'status', 'belum', 'bayar', ',', 'jadi', '2x', 'bayar', '...', 'mohon', 'ditindak', 'lanjut', 'biar', 'gak', 'ada', 'yang', 'seperti', 'itu', 'lagi']
  After  : ['tukar', 'vocer', 'qpon', 'saldo', 'udah', 'kepotong', 'status', 'bayar', 'bayar', 'mohon

---
## Summary

**Data:** Scraped 4,638 QPon reviews from Google Play (`com.qpon.platform`, Indonesian). Saved to `qpon_reviews_raw.csv`.

**Tokenization baselines:**
- English (Gutenberg) — formal, well-structured text
- Indonesian (IndoNLU SMSA) — informal social media, closer to app reviews
- QPon reviews — real project data with slang, code-switching, typos

**Key observations:**
- Rating distribution is heavily polarized: ~44% give 1★, ~44% give 5★
- Indonesian affixation (me-, ber-, di-, -kan, -an) is handled well by Sastrawi stemmer
- Stopword removal significantly reduces noise in short review texts